# 📝 Instrucciones: Análisis de Sentimientos con Naive Bayes

Los modelos **Naive Bayes** son muy útiles cuando queremos analizar sentimientos, clasificar textos en tópicos o recomendaciones, ya que las características de estos desafíos cumplen muy bien con los supuestos teóricos y metodológicos del modelo.

En este proyecto practicarás con un conjunto de datos para crear un clasificador de reseñas de la tienda de Google Play.

---

## Paso 1: Carga del conjunto de datos

El conjunto de datos se encuentra en el siguiente enlace. Puedes cargarlo directamente o descargarlo:

* **URL:** [playstore_reviews.csv](https://raw.githubusercontent.com/4GeeksAcademy/naive-bayes-project-tutorial/main/playstore_reviews.csv)

### Variables del Dataset:
* `package_name`: Nombre de la aplicación móvil (categórico).
* `review`: Comentario sobre la aplicación móvil (categórico).
* `polarity`: Variable de clase (0 o 1), siendo **0 negativo** y **1 positivo** (categórico numérico).

---

## Paso 2: Estudio de variables y su contenido

En este caso, tenemos solo 3 variables. Realmente solo nos interesa la parte del comentario (`review`), ya que la clasificación dependerá de su contenido y no del nombre del paquete. Por lo tanto, la variable `package_name` debe ser eliminada.

### Procesamiento de Texto
Antes de entrenar, debemos procesar el texto plano:

1.  **Limpieza:** Eliminar espacios y convertir a minúsculas.
    ```python
    df["column"] = df["column"].str.strip().str.lower()
    ```
2.  **División de datos:** Separar en `X_train`, `X_test`, `y_train`, `y_test`.
3.  **Vectorización:** Transformar el texto en una matriz de recuento de palabras usando `CountVectorizer`.
    ```python
    vec_model = CountVectorizer(stop_words = "english")
    X_train = vec_model.fit_transform(X_train).toarray()
    X_test = vec_model.transform(X_test).toarray()
    ```

---

## Paso 3: Construye un Naive Bayes

Implementa el modelo eligiendo entre las tres variantes principales según lo estudiado:
* `GaussianNB`
* `MultinomialNB`
* `BernoulliNB`

**Tarea:** Entrena las tres implementaciones y compara cuál es la más adecuada para este tipo de datos (conteo de palabras).

---

## Paso 4: Optimiza el modelo anterior

Tras elegir la mejor variante de Naive Bayes, intenta comparar u optimizar los resultados utilizando un **Random Forest** para ver si mejora la precisión.

---

## Paso 5: Guarda el modelo

Almacena el modelo entrenado en la carpeta correspondiente de tu proyecto (por ejemplo, usando `pickle` o `joblib`).

---

## Paso 6: Explora otras alternativas

¿Qué otros modelos de los que hemos estudiado podrías utilizar para intentar superar los resultados de un Naive Bayes? 
* **Argumenta tu respuesta.**
* **Entrena el modelo elegido.**

## TAREA

In [46]:
#importamos librerias

import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math 
import scipy.stats as stats
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.feature_extraction.text import CountVectorizer
!pip install imblearn
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.1.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [32]:
df = pd.read_csv("../data/raw/playstore_reviews.csv")
df

,package_name,review,polarity
0,com.facebook.katana,privacy at least put some option appear offli...,0
1,com.facebook.katana,"messenger issues ever since the last update, ...",0
2,com.facebook.katana,profile any time my wife or anybody has more ...,0
3,com.facebook.katana,the new features suck for those of us who don...,0
4,com.facebook.katana,forced reload on uploading pic on replying co...,0
...,...,...,...
886,com.rovio.angrybirds,loved it i loooooooooooooovvved it because it...,1
887,com.rovio.angrybirds,all time legendary game the birthday party le...,1
888,com.rovio.angrybirds,ads are way to heavy listen to the bad review...,0
889,com.rovio.angrybirds,fun works perfectly well. ads aren't as annoy...,1


In [33]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   package_name  891 non-null    str  
 1   review        891 non-null    str  
 2   polarity      891 non-null    int64
dtypes: int64(1), str(2)
memory usage: 21.0 KB


In [34]:
df.describe(include='all')

,package_name,review,polarity
count,891,891,891.000000
unique,23,891,NaN
top,com.facebook.katana,privacy at least put some option appear offli...,NaN
freq,40,1,NaN
mean,NaN,NaN,0.344557
std,NaN,NaN,0.475490
min,NaN,NaN,0.000000
25%,NaN,NaN,0.000000
50%,NaN,NaN,0.000000
75%,NaN,NaN,1.000000


In [35]:
df["review"] = df["review"].str.strip().str.lower()
df

,package_name,review,polarity
0,com.facebook.katana,privacy at least put some option appear offlin...,0
1,com.facebook.katana,"messenger issues ever since the last update, i...",0
2,com.facebook.katana,profile any time my wife or anybody has more t...,0
3,com.facebook.katana,the new features suck for those of us who don'...,0
4,com.facebook.katana,forced reload on uploading pic on replying com...,0
...,...,...,...
886,com.rovio.angrybirds,loved it i loooooooooooooovvved it because it ...,1
887,com.rovio.angrybirds,all time legendary game the birthday party lev...,1
888,com.rovio.angrybirds,ads are way to heavy listen to the bad reviews...,0
889,com.rovio.angrybirds,fun works perfectly well. ads aren't as annoyi...,1


In [36]:
df.drop('package_name', axis=1, inplace=True)

In [37]:
df

,review,polarity
0,privacy at least put some option appear offlin...,0
1,"messenger issues ever since the last update, i...",0
2,profile any time my wife or anybody has more t...,0
3,the new features suck for those of us who don'...,0
4,forced reload on uploading pic on replying com...,0
...,...,...
886,loved it i loooooooooooooovvved it because it ...,1
887,all time legendary game the birthday party lev...,1
888,ads are way to heavy listen to the bad reviews...,0
889,fun works perfectly well. ads aren't as annoyi...,1


In [40]:
X = df.drop('polarity',axis=1)
y = df['polarity']

In [53]:
y.value_counts()

polarity
0    584
1    307
Name: count, dtype: int64

In [47]:
ros = RandomOverSampler(sampling_strategy="minority",random_state=42)
X_over, y_over = ros.fit_resample(X, y)

In [52]:
X_train, X_test, y_train, y_test = train_test_split(X_over, y_over, test_size=0.33, random_state=42)

In [51]:
vec_model = CountVectorizer(stop_words = "english")
X_train = vec_model.fit_transform(X_train).toarray()
X_test = vec_model.transform(X_test).toarray()

In [ ]:
model = gaussianNB()
model.fit(X_train, y_train)

NameError: name 'gai' is not defined